In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
import joblib

RAW = Path("../Dataset/Raw")
PROCESSED = Path("../Dataset/Processed")
REPORTS = Path("../Reports")
REPORTS.mkdir(parents=True, exist_ok=True)

print("RAW:", RAW.resolve())
print("PROCESSED:", PROCESSED.resolve())

RAW: C:\CSIE\My Projects\Data_Science\BoardGames_Analyzer\Dataset\Raw
PROCESSED: C:\CSIE\My Projects\Data_Science\BoardGames_Analyzer\Dataset\Processed


In [2]:
meta_path = RAW / "games_detailed_info.csv"
df_meta = pd.read_csv(meta_path, low_memory=False)

if "id" not in df_meta.columns:
    raise ValueError("games_detailed_info.csv must contain column 'id'")

df_meta = df_meta.set_index("id")

# Ensure a final display name column
if "name" not in df_meta.columns:
    if "primary" in df_meta.columns:
        df_meta["name"] = df_meta["primary"]
    elif "Name" in df_meta.columns:
        df_meta["name"] = df_meta["Name"]
    else:
        df_meta["name"] = ""

# Cast index to int safely
df_meta.index = df_meta.index.astype(int)

print("Items:", len(df_meta))
df_meta[["name"]].head()

Items: 21631


,name
id,
30549,Pandemic
822,Carcassonne
13,Catan
68448,7 Wonders
36218,Dominion


In [3]:
NAME_COL = "name"
MECH_COL = "boardgamemechanic"
CAT_COL  = "boardgamecategory"      # used for tags like Mystery/Horror/Farming
FAM_COL  = "boardgamefamily"        # optional (not required for all)
PUB_COL  = "boardgamepublisher"

NUM_COLS = {
    "minplayers": "minplayers",
    "maxplayers": "maxplayers",
    "minplaytime": "minplaytime",
    "maxplaytime": "maxplaytime",
    "minage": "minage",
    "avgweight": "averageweight",   # difficulty proxy
    "usersrated": "usersrated",     # optional ranking tie-breaker
}

# sanity checks (don't crash if missing, but warn)
for c in [MECH_COL, CAT_COL, FAM_COL, PUB_COL]:
    if c not in df_meta.columns:
        print(f"WARNING: missing column in df_meta: {c}")

df_meta.loc[[13, 30549, 822], [NAME_COL, MECH_COL, CAT_COL, FAM_COL, PUB_COL]].head()

,name,boardgamemechanic,boardgamecategory,boardgamefamily,boardgamepublisher
id,,,,,
13,Catan,"['Dice Rolling', 'Hexagon Grid', 'Income', 'Mo...","['Economic', 'Negotiation']","['Animals: Sheep', 'Components: Hexagonal Tile...","['KOSMOS', '999 Games', 'Albi', 'Asmodee', 'As..."
30549,Pandemic,"['Action Points', 'Cooperative Game', 'Hand Ma...",['Medical'],"['Components: Map (Global Scale)', 'Components...","['Z-Man Games', 'Albi', 'Asmodee', 'Asmodee It..."
822,Carcassonne,"['Area Majority / Influence', 'Map Addition', ...","['City Building', 'Medieval', 'Territory Build...","['Cities: Carcassonne (France)', 'Components: ...","['Hans im Glück', '999 Games', 'Albi', 'Bard C..."


In [4]:
def norm_token(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip().lower())

def to_tokens(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [norm_token(t) for t in x if str(t).strip()]
    s = str(x).strip()
    if not s:
        return []
    # Convert "['A', 'B']" into "A, B"
    s = s.replace("[", "").replace("]", "").replace("'", "").replace('"', "")
    # Split by common delimiters
    if "|" in s:
        parts = s.split("|")
    elif "," in s:
        parts = s.split(",")
    else:
        parts = [s]
    return [norm_token(p) for p in parts if str(p).strip()]

def get_tokens_series(df, col):
    if col not in df.columns:
        return pd.Series([[]]*len(df), index=df.index)
    return df[col].apply(to_tokens)

mech_tokens = get_tokens_series(df_meta, MECH_COL)
cat_tokens  = get_tokens_series(df_meta, CAT_COL)
fam_tokens  = get_tokens_series(df_meta, FAM_COL)
pub_tokens  = get_tokens_series(df_meta, PUB_COL)

# quick check
for gid in [13, 30549, 822]:
    print(gid, df_meta.loc[gid, "name"])
    print("  mech:", mech_tokens.loc[gid][:8])
    print("  cat :", cat_tokens.loc[gid][:8])

13 Catan
  mech: ['dice rolling', 'hexagon grid', 'income', 'modular board', 'network and route building', 'race', 'random production', 'trading']
  cat : ['economic', 'negotiation']
30549 Pandemic
  mech: ['action points', 'cooperative game', 'hand management', 'point to point movement', 'set collection', 'trading', 'variable player powers']
  cat : ['medical']
822 Carcassonne
  mech: ['area majority / influence', 'map addition', 'tile placement']
  cat : ['city building', 'medieval', 'territory building']


In [5]:
desc_sim = joblib.load(PROCESSED / "desc_topk_csr.joblib")
content_sim = joblib.load(PROCESSED / "content_topk_csr.joblib")
emb_sim = joblib.load(PROCESSED / "emb_topk_csr.joblib")

meta_ids = df_meta.index.to_numpy()
id_to_idx = {int(gid): i for i, gid in enumerate(meta_ids)}
idx_to_id = {i: int(gid) for i, gid in enumerate(meta_ids)}

print("Similarity matrices loaded.")
print("Universe size:", len(meta_ids))

Similarity matrices loaded.
Universe size: 21631


In [6]:
sent_path = PROCESSED / "sentiment_summary.csv"
sent_summary = pd.read_csv(sent_path, index_col="id") if sent_path.exists() else None

sent_mean = None
if sent_summary is not None and "avg_sentiment_score" in sent_summary.columns:
    sent_summary.index = sent_summary.index.astype(int)
    sent_mean = float(sent_summary["avg_sentiment_score"].mean())

print("Sentiment loaded:", sent_summary is not None)

Sentiment loaded: True


In [7]:
ratings_path = RAW / "large_user_ratings.csv"
ratings_df = pd.read_csv(ratings_path)

USER_COL = "Username"
ITEM_COL = "BGGId"
RATING_COL = "Rating"

ratings_df = ratings_df.dropna(subset=[USER_COL, ITEM_COL, RATING_COL]).copy()
ratings_df[ITEM_COL] = ratings_df[ITEM_COL].astype(int)

pop_counts = ratings_df.groupby(ITEM_COL).size()
pop_counts = pop_counts.reindex(meta_ids).fillna(0).astype(int)

# normalize 0..1
pop_norm = (pop_counts - pop_counts.min()) / (pop_counts.max() - pop_counts.min() + 1e-9)
pop_norm = pop_norm.fillna(0.0)

print("Popularity ready.")

Popularity ready.


In [8]:
def safe_minmax(series, full_index):
    s = series.reindex(full_index).astype(float)
    arr = s.values
    if np.all(np.isnan(arr)):
        return pd.Series(0.0, index=full_index)
    mn = np.nanmin(arr)
    mx = np.nanmax(arr)
    if mx == mn:
        return pd.Series(0.0, index=full_index)
    arr = np.nan_to_num(arr, nan=np.nanmean(arr))
    norm = (arr - mn) / (mx - mn + 1e-9)
    return pd.Series(norm, index=full_index)

def topk_from_sim(seed_ids, sim_csr, k=1200, strategy="mean"):
    seed_idxs = [id_to_idx[sid] for sid in seed_ids if sid in id_to_idx]
    if not seed_idxs:
        return pd.Series(dtype=float)

    scores = defaultdict(float)
    counts = defaultdict(int)

    for si in seed_idxs:
        row = sim_csr.getrow(si)
        for c, v in zip(row.indices, row.data):
            gid = idx_to_id[c]
            if strategy == "max":
                scores[gid] = max(scores[gid], float(v))
            else:
                scores[gid] += float(v)
                counts[gid] += 1

    if strategy != "max":
        for gid in list(scores.keys()):
            scores[gid] /= max(counts[gid], 1)

    for sid in seed_ids:
        scores.pop(int(sid), None)

    items = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    return pd.Series(dict(items), dtype=float)

In [9]:
df_names = df_meta[[NAME_COL]].copy()
df_names["name_norm"] = df_names[NAME_COL].fillna("").apply(norm_token)

def search_titles(query, topn=10):
    q = norm_token(query)
    if not q:
        return pd.DataFrame(columns=["id", "name"])
    # exact match first
    exact = df_names[df_names["name_norm"] == q].reset_index()[["id", NAME_COL]]
    if len(exact) > 0:
        exact.columns = ["id", "name"]
        return exact.head(topn)

    # contains match
    cand = df_names[df_names["name_norm"].str.contains(re.escape(q), na=False)].reset_index()[["id", NAME_COL]]
    cand.columns = ["id", "name"]

    # rank by popularity/usersrated if available
    if "usersrated" in df_meta.columns:
        cand = cand.merge(df_meta[["usersrated"]], left_on="id", right_index=True, how="left")
        cand["usersrated"] = pd.to_numeric(cand["usersrated"], errors="coerce").fillna(0)
        cand = cand.sort_values("usersrated", ascending=False).drop(columns=["usersrated"])
    else:
        cand = cand.copy()

    return cand.head(topn)

def resolve_titles_to_ids(titles, pick="auto"):
    """
    titles: list[str]
    pick:
      - "auto": pick best match (most usersrated) if ambiguous
      - "all": return all matches (can be messy)
    """
    seed_ids = []
    candidates_info = []

    for t in titles:
        matches = search_titles(t, topn=10)
        if len(matches) == 0:
            candidates_info.append((t, []))
            continue

        ids = matches["id"].tolist()
        candidates_info.append((t, ids))

        if pick == "all":
            seed_ids.extend(ids)
        else:  # auto: pick top-1
            seed_ids.append(int(ids[0]))

    # dedupe keep order
    seen = set()
    uniq = []
    for x in seed_ids:
        if int(x) not in seen:
            uniq.append(int(x))
            seen.add(int(x))

    return uniq, candidates_info

# quick test
search_titles("splendor", topn=5)

,id,name
0,148228,Splendor


In [10]:
DIFF_BUCKETS = {
    "low": (None, 2.0),
    "medium": (2.0, 3.0),
    "high": (3.0, None),
}

# Keywords -> mechanics/categories
# You can expand this list anytime.
TYPE_SYNONYMS = {
    "co-op": {"mechanic": ["cooperative game"]},
    "coop": {"mechanic": ["cooperative game"]},
    "cooperative": {"mechanic": ["cooperative game"]},
    "competitive": {"exclude_mechanic": ["cooperative game"]},  # simple interpretation
    "mystery": {"category": ["mystery"]},
    "horror": {"category": ["horror"]},
    "farming": {"category": ["farming"]},
    "fun": {"category": ["party game", "humor"]},  # weak mapping, optional
    "family": {"family": ["family games", "children's games"]},
}

def parse_user_types(type_inputs):
    """
    type_inputs: list[str] (free text)
    returns: dict with keys:
      include_mech, exclude_mech, include_cat, include_family, difficulty_bucket
    """
    include_mech, exclude_mech, include_cat, include_family = [], [], [], []
    difficulty = None

    for raw in (type_inputs or []):
        t = norm_token(raw)

        if t in ("low", "medium", "high"):
            difficulty = t
            continue

        rule = TYPE_SYNONYMS.get(t)
        if rule:
            include_mech += rule.get("mechanic", [])
            exclude_mech += rule.get("exclude_mechanic", [])
            include_cat += rule.get("category", [])
            include_family += rule.get("family", [])
        else:
            # fallback: treat unknown token as category tag
            include_cat.append(t)

    # dedupe
    def dedupe(lst):
        out, seen = [], set()
        for x in lst:
            x = norm_token(x)
            if x and x not in seen:
                out.append(x); seen.add(x)
        return out

    return {
        "include_mech": dedupe(include_mech),
        "exclude_mech": dedupe(exclude_mech),
        "include_cat": dedupe(include_cat),
        "include_family": dedupe(include_family),
        "difficulty": difficulty
    }

In [11]:
def recommend_by_preferences(
    types=None,
    min_players=None, max_players=None,
    min_playtime=None, max_playtime=None,
    max_age=None, min_age=None,
    topk=20,
    w_trait=0.65, w_sent=0.10, w_pop=0.25,
    require_any=True  # if True: must match at least one included token
):
    rules = parse_user_types(types)
    inc_mech = set(rules["include_mech"])
    exc_mech = set(rules["exclude_mech"])
    inc_cat  = set(rules["include_cat"])
    inc_fam  = set(rules["include_family"])
    diff = rules["difficulty"]

    candidates = df_meta.copy()

    # numeric filters (if columns exist)
    def apply_range(col, lo=None, hi=None):
        nonlocal candidates
        if col not in candidates.columns:
            return
        s = pd.to_numeric(candidates[col], errors="coerce")
        mask = pd.Series(True, index=candidates.index)
        if lo is not None:
            mask &= (s >= lo)
        if hi is not None:
            mask &= (s <= hi)
        candidates = candidates[mask.fillna(False)]

    apply_range(NUM_COLS["minplayers"], min_players, None)
    apply_range(NUM_COLS["maxplayers"], None, max_players)
    apply_range(NUM_COLS["minplaytime"], min_playtime, None)
    apply_range(NUM_COLS["maxplaytime"], None, max_playtime)
    apply_range(NUM_COLS["minage"], min_age, None)
    apply_range(NUM_COLS["minage"], None, max_age)

    # difficulty mapping via averageweight
    if diff in DIFF_BUCKETS and NUM_COLS["avgweight"] in candidates.columns:
        lo, hi = DIFF_BUCKETS[diff]
        apply_range(NUM_COLS["avgweight"], lo, hi)

    cand_ids = candidates.index.astype(int)

    rows = []
    for gid in cand_ids:
        m = set(mech_tokens.loc[gid])
        c = set(cat_tokens.loc[gid])
        f = set(fam_tokens.loc[gid])

        # exclude mechanics
        if exc_mech and len(m & exc_mech) > 0:
            continue

        mech_hit = len(m & inc_mech) if inc_mech else 0
        cat_hit  = len(c & inc_cat) if inc_cat else 0
        fam_hit  = len(f & inc_fam) if inc_fam else 0

        total_prefs = len(inc_mech) + len(inc_cat) + len(inc_fam)
        if total_prefs == 0:
            score_trait = 0.0
        else:
            score_trait = (mech_hit + cat_hit + fam_hit) / total_prefs

        if require_any and total_prefs > 0 and (mech_hit + cat_hit + fam_hit) == 0:
            continue

        rows.append((gid, score_trait, mech_hit, cat_hit, fam_hit))

    if len(rows) == 0:
        return pd.DataFrame(columns=["game_id","name","score"])

    trait_df = pd.DataFrame(rows, columns=["game_id","score_trait","mech_hit","cat_hit","fam_hit"]).set_index("game_id")
    trait_norm = safe_minmax(trait_df["score_trait"], trait_df.index.to_numpy())

    # sentiment
    if sent_summary is not None and "avg_sentiment_score" in sent_summary.columns:
        sent = sent_summary["avg_sentiment_score"].reindex(trait_df.index).fillna(sent_mean)
        sent_norm = safe_minmax(sent, trait_df.index.to_numpy())
    else:
        sent_norm = pd.Series(0.0, index=trait_df.index)

    # popularity
    pop = pop_norm.reindex(trait_df.index).fillna(0.0)

    final = w_trait*trait_norm + w_sent*sent_norm + w_pop*pop
    top_ids = final.sort_values(ascending=False).head(topk).index.astype(int).tolist()

    out = []
    for gid in top_ids:
        m = set(mech_tokens.loc[gid])
        c = set(cat_tokens.loc[gid])
        f = set(fam_tokens.loc[gid])

        out.append({
            "game_id": gid,
            "name": df_meta.loc[gid, NAME_COL] if gid in df_meta.index else "",
            "score": float(final.loc[gid]),
            "score_trait": float(trait_norm.loc[gid]),
            "score_sent": float(sent_norm.loc[gid]),
            "score_pop": float(pop.loc[gid]),
            "matched_mechanics": ", ".join(sorted(list(m & inc_mech))[:8]),
            "matched_categories": ", ".join(sorted(list(c & inc_cat))[:8]),
            "matched_family": ", ".join(sorted(list(f & inc_fam))[:8]),
        })

    return pd.DataFrame(out).sort_values("score", ascending=False)

In [12]:
def recommend_like_titles(
    titles,
    topk=20,
    w_text=0.35, w_emb=0.35, w_sent=0.10, w_pop=0.20,
    candidate_k=1200,
    pick="auto"
):
    seed_ids, candidates_info = resolve_titles_to_ids(titles, pick=pick)
    seed_ids = [sid for sid in seed_ids if sid in id_to_idx]

    if len(seed_ids) == 0:
        return pd.DataFrame({
            "note": ["No seed titles found. Try a different spelling."],
            "candidates_info": [str(candidates_info)]
        })

    desc_s = topk_from_sim(seed_ids, desc_sim, k=candidate_k).reindex(meta_ids).fillna(0.0)
    cont_s = topk_from_sim(seed_ids, content_sim, k=candidate_k).reindex(meta_ids).fillna(0.0)
    emb_s  = topk_from_sim(seed_ids, emb_sim, k=candidate_k).reindex(meta_ids).fillna(0.0)

    text_norm = safe_minmax(0.5*desc_s + 0.5*cont_s, meta_ids)
    emb_norm  = safe_minmax(emb_s, meta_ids)

    if sent_summary is not None and "avg_sentiment_score" in sent_summary.columns:
        sent = sent_summary["avg_sentiment_score"].reindex(meta_ids).fillna(sent_mean)
        sent_norm = safe_minmax(sent, meta_ids)
    else:
        sent_norm = pd.Series(0.0, index=meta_ids)

    pop = pop_norm.reindex(meta_ids).fillna(0.0)

    score = w_text*text_norm + w_emb*emb_norm + w_sent*sent_norm + w_pop*pop

    # remove seeds
    for sid in seed_ids:
        score.loc[sid] = -1

    rec_ids = score.sort_values(ascending=False).head(topk).index.astype(int).tolist()

    # explanation tokens from union of seeds
    seed_mech = set().union(*[set(mech_tokens.loc[sid]) for sid in seed_ids])
    seed_cat  = set().union(*[set(cat_tokens.loc[sid]) for sid in seed_ids])
    seed_pub  = set().union(*[set(pub_tokens.loc[sid]) for sid in seed_ids])

    out = []
    for gid in rec_ids:
        m = set(mech_tokens.loc[gid])
        c = set(cat_tokens.loc[gid])
        p = set(pub_tokens.loc[gid])

        out.append({
            "game_id": gid,
            "name": df_meta.loc[gid, NAME_COL] if gid in df_meta.index else "",
            "score": float(score.loc[gid]),
            "score_text": float(text_norm.loc[gid]),
            "score_emb": float(emb_norm.loc[gid]),
            "score_sent": float(sent_norm.loc[gid]),
            "score_pop": float(pop.loc[gid]),
            "matched_mechanics": ", ".join(sorted(list(m & seed_mech))[:8]),
            "matched_categories": ", ".join(sorted(list(c & seed_cat))[:8]),
            "matched_publishers": ", ".join(sorted(list(p & seed_pub))[:6]),
            "seed_ids_used": seed_ids,
            "seed_title_candidates": str(candidates_info)[:200] + ("..." if len(str(candidates_info)) > 200 else "")
        })

    return pd.DataFrame(out).sort_values("score", ascending=False)

In [13]:
def discover(
    titles=None,
    types=None,
    topk=20,
    pref_pool=800,
    pick="auto"
):
    titles = titles or []
    types = types or []

    # 1) preference pool (filter + rank)
    pref_df = recommend_by_preferences(types=types, topk=pref_pool, require_any=True)

    # if no titles, return pure preference results
    if len(titles) == 0:
        return pref_df.head(topk)

    # if preference filter empties out, fallback to pure title-like
    if pref_df.empty:
        return recommend_like_titles(titles, topk=topk, pick=pick)

    # 2) similarity ranking from titles
    sim_df = recommend_like_titles(titles, topk=2000, pick=pick)
    if "game_id" not in sim_df.columns:
        return sim_df  # contains note/errors

    sim_df = sim_df.set_index("game_id")
    pref_df = pref_df.set_index("game_id")

    # keep only preference candidates
    merged = pref_df.join(sim_df[["score","score_text","score_emb","score_sent","score_pop",
                                 "matched_mechanics","matched_categories","matched_publishers",
                                 "seed_ids_used","seed_title_candidates"]],
                          how="left", rsuffix="_like")

    merged["score_like"] = merged["score"].fillna(0.0)

    # combine: 65% similarity + 35% preference fit
    merged["score_final"] = 0.65*merged["score_like"] + 0.35*merged["score_trait"]

    out = merged.sort_values("score_final", ascending=False).head(topk).reset_index()
    out = out.rename(columns={"score_final": "score"})
    return out

In [14]:
recommend_by_preferences(types=["Mystery", "Horror", "Medium"], topk=10)

recommend_like_titles(["Splendor", "Azul"], topk=10)

discover(
    titles=["Splendor", "Azul"],
    types=["Mystery", "Horror", "Medium"],
    topk=10
)

,game_id,name,score,score_trait,score_sent,score_pop,matched_mechanics,matched_categories,matched_family,score_like,score_text,score_emb,score_sent_like,score_pop_like,matched_mechanics_like,matched_categories_like,matched_publishers,seed_ids_used,seed_title_candidates,score
0,10547,Betrayal at House on the Hill,0.091005,0.0,0.0,0.364022,,horror,,0.091005,0.0,0.0,0.509526,0.364022,,,asmodee,"[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]",0.059154
1,205059,Mansions of Madness: Second Edition,0.065077,0.0,0.0,0.260310,,horror,,0.065077,0.0,0.0,0.509526,0.260310,,puzzle,"asmodee china, asterion press, galápagos jogos...","[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]",0.042300
2,100423,Elder Sign,0.053422,0.0,0.0,0.213688,,horror,,0.053422,0.0,0.0,0.509526,0.213688,,card game,galápagos jogos,"[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]",0.034724
3,37046,Ghost Stories,0.046256,0.0,0.0,0.185022,,horror,,0.046256,0.0,0.0,0.509526,0.185022,,,"asmodee, lautapelit.fi, rebel sp. z o.o.","[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]",0.030066
4,113924,Zombicide,0.039999,0.0,0.0,0.159994,,horror,,0.039999,0.0,0.0,0.509526,0.159994,,,"asmodee, asterion press, galápagos jogos","[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]",0.025999
5,29368,Last Night on Earth: The Zombie Game,0.031932,0.0,0.0,0.127728,,horror,,0.031932,0.0,0.0,0.509526,0.127728,,,,"[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]",0.020756
6,176189,Zombicide: Black Plague,0.031389,0.0,0.0,0.125557,,horror,,0.031389,0.0,0.0,0.509526,0.125557,,,"asterion press, galápagos jogos","[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]",0.020403
7,146652,Legendary Encounters: An Alien Deck Building Game,0.027494,0.0,0.0,0.109976,,horror,,0.027494,0.0,0.0,0.509526,0.109976,card drafting,card game,,"[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]",0.017871
8,282524,Horrified,0.021485,0.0,0.0,0.085941,,horror,,0.021485,0.0,0.0,0.509526,0.085941,,,,"[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]",0.013965
9,20963,Fury of Dracula (Second Edition),0.021035,0.0,0.0,0.084141,,horror,,0.021035,0.0,0.0,0.509526,0.084141,,,,"[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]",0.013673
